In [1]:
!pip install -q torch \
 transformers \
 bitsandbytes \
 nvidia-ml-py \
 langchain-groq \
 langchain-google-genai \
 langchain-mistralai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.5/70.5 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 5.3 MB/s eta 0:00:00


In [2]:
!pip install -q --upgrade transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 50.1 MB/s eta 0:00:00


In [3]:
import torch
import time
import re
import threading
import pandas as pd
import pynvml
from google.colab import userdata
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

from langchain_groq import ChatGroq
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_mistralai import ChatMistralAI
from langchain.messages import HumanMessage

print("Imports successful")

Imports successful


In [ ]:
! nvidia-smi

Sat Jun 27 09:00:28 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
df = pd.read_json('benchmark.json')
print(df["prompt"][1])

If a clock strikes 6 times in 5 seconds, how many seconds will it take to strike 12 times?


In [ ]:
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",                      # Use NF4 data type for higher precision
    bnb_4bit_use_double_quant=True,                 # Nested quantization to save extra VRAM
    bnb_4bit_compute_dtype=torch.bfloat16           # Speeds up processing if GPU supports it
)

In [ ]:
tokenizer_deepseek = AutoTokenizer.from_pretrained("deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B")
model_deepseek = AutoModelForCausalLM.from_pretrained("deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B",
                                             dtype=torch.float16,
                                            quantization_config = quantization_config,
                                             device_map ='auto')

config.json:   0%|          | 0.00/679 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.07k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.55G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

In [ ]:

# ── Quantization config ──────────────────────────────────────────────
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

# ── Load model ───────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained("deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B")
model = AutoModelForCausalLM.from_pretrained(
    "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B",
    quantization_config=quantization_config,
    device_map='auto'
)

# ── GPU monitor setup ────────────────────────────────────────────────
pynvml.nvmlInit()
handle = pynvml.nvmlDeviceGetHandleByIndex(0)

# ── Helper: poll GPU in background thread during generation ──────────
def poll_gpu(handle, interval, results, stop_event):
    """Polls GPU every `interval` seconds until stop_event is set."""
    util_readings, temp_readings, mem_readings = [], [], []
    while not stop_event.is_set():
        util_readings.append(pynvml.nvmlDeviceGetUtilizationRates(handle).gpu)
        temp_readings.append(pynvml.nvmlDeviceGetTemperature(handle, pynvml.NVML_TEMPERATURE_GPU))
        mem_readings.append(pynvml.nvmlDeviceGetMemoryInfo(handle).used / 1024**2)
        time.sleep(interval)
    results['gpu_usage_pct']   = round(max(util_readings), 2)   # peak usage
    results['gpu_temp_c']      = round(max(temp_readings), 2)   # peak temp
    results['gpu_mem_used_mb'] = round(max(mem_readings), 2)    # peak memory

# ── Strip DeepSeek <think> chain-of-thought block ───────────────────
def strip_think_block(text):
    return re.sub(r'<think>.*?</think>', '', str(text), flags=re.DOTALL).strip()

# ── Main inference loop ──────────────────────────────────────────────
deepseek_records = []

for i in range(len(df)):
    inputs = tokenizer(
        df["prompt"][i],
        return_tensors='pt'
    ).to(model.device)

    prompt_tokens = inputs["input_ids"].shape[1]

    # Start background GPU polling
    gpu_results = {}
    stop_event  = threading.Event()
    gpu_thread  = threading.Thread(
        target=poll_gpu,
        args=(handle, 0.5, gpu_results, stop_event)  # poll every 0.5s
    )
    gpu_thread.start()

    # Generation
    start = time.time()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=2048,
            pad_token_id=tokenizer.eos_token_id   # avoids warning on Qwen
        )
    latency = time.time() - start

    # Stop GPU polling
    stop_event.set()
    gpu_thread.join()

    # Decode only new tokens (strips prompt echo)
    new_tokens    = outputs[0][prompt_tokens:]
    output_tokens = new_tokens.shape[0]
    text          = tokenizer.decode(new_tokens, skip_special_tokens=True)
    text          = strip_think_block(text)

    deepseek_records.append({
        "question_id"     : i,
        "question"        : df["prompt"][i],
        "answer"          : text,
        "prompt_tokens"   : prompt_tokens,
        "output_tokens"   : output_tokens,
        "total_tokens"    : prompt_tokens + output_tokens,
        "latency_sec"     : round(latency, 4),
        "tokens_per_sec"  : round(output_tokens / latency, 2),
        "gpu_usage_pct"   : gpu_results.get('gpu_usage_pct'),
        "gpu_temp_c"      : gpu_results.get('gpu_temp_c'),
        "gpu_mem_used_mb" : gpu_results.get('gpu_mem_used_mb'),
    })

    print(f"[{i+1}/{len(df)}] done | "
          f"out_tokens: {output_tokens} | "
          f"latency: {latency:.2f}s | "
          f"tok/s: {output_tokens/latency:.1f} | "
          f"gpu: {gpu_results.get('gpu_usage_pct')}% | "
          f"temp: {gpu_results.get('gpu_temp_c')}C | "
          f"mem: {gpu_results.get('gpu_mem_used_mb')}MB")

deepseek_df = pd.DataFrame(deepseek_records)
deepseek_df['model'] = 'deepseek'
deepseek_df.to_csv('deepseek_metrics.csv', index=False)
print("\nSaved deepseek_metrics.csv")
print(deepseek_df[['question_id','output_tokens','latency_sec',
                    'tokens_per_sec','gpu_usage_pct','gpu_temp_c',
                    'gpu_mem_used_mb']].to_string())

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


[1/30] done | out_tokens: 2032 | latency: 160.16s | tok/s: 12.7 | gpu: 47% | temp: 58C | mem: 3786.19MB
[2/30] done | out_tokens: 2048 | latency: 156.53s | tok/s: 13.1 | gpu: 47% | temp: 59C | mem: 3786.19MB
[3/30] done | out_tokens: 533 | latency: 40.74s | tok/s: 13.1 | gpu: 42% | temp: 59C | mem: 3786.19MB
[4/30] done | out_tokens: 2048 | latency: 155.76s | tok/s: 13.1 | gpu: 47% | temp: 59C | mem: 3786.19MB
[5/30] done | out_tokens: 1309 | latency: 100.65s | tok/s: 13.0 | gpu: 48% | temp: 59C | mem: 3786.19MB
[6/30] done | out_tokens: 265 | latency: 20.08s | tok/s: 13.2 | gpu: 44% | temp: 56C | mem: 3786.19MB
[7/30] done | out_tokens: 577 | latency: 45.34s | tok/s: 12.7 | gpu: 49% | temp: 54C | mem: 3786.19MB
[8/30] done | out_tokens: 175 | latency: 13.68s | tok/s: 12.8 | gpu: 40% | temp: 54C | mem: 3786.19MB
[9/30] done | out_tokens: 1202 | latency: 92.64s | tok/s: 13.0 | gpu: 47% | temp: 54C | mem: 3786.19MB
[10/30] done | out_tokens: 1634 | latency: 125.31s | tok/s: 13.0 | gpu: 4

In [ ]:
deepseek_df.head()

,question_id,question,answer,prompt_tokens,output_tokens,total_tokens,latency_sec,tokens_per_sec,gpu_usage_pct,gpu_temp_c,gpu_mem_used_mb,model
0,0,A train leaves Station A traveling at 60 mph. ...,"To solve this problem, we need to determine th...",50,2032,2082,160.1579,12.69,47,58,3786.19,deepseek
1,1,"If a clock strikes 6 times in 5 seconds, how m...","To solve this, I need to figure out the relati...",25,2048,2073,156.5330,13.08,47,59,3786.19,deepseek
2,2,All prompt engineers are tech enthusiasts. Som...,"Let me think carefully.\n\nFirst, the first pr...",37,533,570,40.7378,13.08,42,59,3786.19,deepseek
3,3,A bat and a ball cost $1.10 in total. The bat ...,"Let me think about this step by step.\nFirst, ...",34,2048,2082,155.7587,13.15,47,59,3786.19,deepseek
4,4,If 5 machines take 5 minutes to make 5 widgets...,Assume each machine works independently and at...,32,1309,1341,100.6547,13.00,48,59,3786.19,deepseek


In [6]:
llm_groq = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.4,
    api_key=userdata.get("GROQ_API_KEY")
)

groq_records = []

for i in range(len(df)):
    start = time.time()
    output = llm_groq.invoke(df["prompt"][i])
    latency = time.time() - start

    # Groq gives token usage directly in metadata
    usage = output.response_metadata.get('token_usage', {})
    prompt_tokens    = usage.get('prompt_tokens', 0)
    output_tokens    = usage.get('completion_tokens', 0)
    total_tokens     = usage.get('total_tokens', 0)
    # use groq's own time if available, else use ours
    groq_latency     = usage.get('completion_time', latency)

    groq_records.append({
        "question_id"      : i,
        "question"         : df["prompt"][i],
        "answer"           : output.content,
        "prompt_tokens"    : prompt_tokens,
        "output_tokens"    : output_tokens,
        "total_tokens"     : total_tokens,
        "latency_sec"      : round(groq_latency, 4),
        "tokens_per_sec"   : round(output_tokens / groq_latency, 2) if groq_latency > 0 else 0,
        "gpu_usage_pct"    : None,   # cloud API — not applicable
        "gpu_temp_c"       : None,
        "gpu_mem_used_mb"  : None,
    })

    print(f"[{i+1}/{len(df)}] done | tokens: {total_tokens} | latency: {groq_latency:.2f}s")

groq_df = pd.DataFrame(groq_records)
groq_df['model'] = 'groq/llama-3.1-8b-instant'
groq_df.to_csv('groq_metrics.csv', index=False)
print(groq_df.head())

[1/30] done | tokens: 256 | latency: 0.23s
[2/30] done | tokens: 196 | latency: 0.18s
[3/30] done | tokens: 373 | latency: 0.58s
[4/30] done | tokens: 206 | latency: 0.23s
[5/30] done | tokens: 146 | latency: 0.09s
[6/30] done | tokens: 266 | latency: 0.23s
[7/30] done | tokens: 297 | latency: 0.26s
[8/30] done | tokens: 143 | latency: 0.13s
[9/30] done | tokens: 222 | latency: 0.19s
[10/30] done | tokens: 277 | latency: 0.22s
[11/30] done | tokens: 300 | latency: 0.26s
[12/30] done | tokens: 485 | latency: 0.71s
[13/30] done | tokens: 511 | latency: 0.63s
[14/30] done | tokens: 427 | latency: 0.41s
[15/30] done | tokens: 653 | latency: 0.70s
[16/30] done | tokens: 94 | latency: 0.13s
[17/30] done | tokens: 561 | latency: 0.95s
[18/30] done | tokens: 541 | latency: 0.62s
[19/30] done | tokens: 242 | latency: 0.24s
[20/30] done | tokens: 785 | latency: 0.87s
[21/30] done | tokens: 181 | latency: 0.29s
[22/30] done | tokens: 89 | latency: 0.04s
[23/30] done | tokens: 118 | latency: 0.18s

In [7]:
llm_mistral = ChatMistralAI(
    model="mistral-small-2506",
    temperature=0.4,
    max_tokens=1024,
    api_key=userdata.get("MISTRAL_API_KEY")
)

mistral_records = []

for i in range(len(df)):
    start = time.time()
    output = llm_mistral.invoke(df["prompt"][i])
    latency = time.time() - start

    # Debug on first row
    if i == 0:
        print("usage_metadata:", output.usage_metadata)

    # Same as Groq — usage_metadata is top level on output object
    prompt_tokens = output.usage_metadata.get('input_tokens', 0)
    output_tokens = output.usage_metadata.get('output_tokens', 0)
    total_tokens  = output.usage_metadata.get('total_tokens', 0)

    mistral_records.append({
        "question_id"     : i,
        "question"        : df["prompt"][i],
        "answer"          : output.content,
        "prompt_tokens"   : prompt_tokens,
        "output_tokens"   : output_tokens,
        "total_tokens"    : total_tokens,
        "latency_sec"     : round(latency, 4),
        "tokens_per_sec"  : round(output_tokens / latency, 2) if latency > 0 else 0,
        "gpu_usage_pct"   : None,
        "gpu_temp_c"      : None,
        "gpu_mem_used_mb" : None,
    })

    print(f"[{i+1}/{len(df)}] done | tokens: {total_tokens} | latency: {latency:.2f}s")

mistral_df = pd.DataFrame(mistral_records)
mistral_df['model'] = 'mistral-small-2506'
mistral_df.to_csv('mistral_metrics.csv', index=False)
print(mistral_df.head())

usage_metadata: {'input_tokens': 53, 'output_tokens': 1024, 'total_tokens': 1077}
[1/30] done | tokens: 1077 | latency: 5.86s
[2/30] done | tokens: 934 | latency: 5.37s
[3/30] done | tokens: 1068 | latency: 6.16s
[4/30] done | tokens: 737 | latency: 4.82s
[5/30] done | tokens: 1059 | latency: 5.95s
[6/30] done | tokens: 1065 | latency: 5.71s
[7/30] done | tokens: 368 | latency: 2.17s
[8/30] done | tokens: 168 | latency: 1.27s
[9/30] done | tokens: 1013 | latency: 6.52s
[10/30] done | tokens: 515 | latency: 3.25s
[11/30] done | tokens: 323 | latency: 1.90s
[12/30] done | tokens: 402 | latency: 2.40s
[13/30] done | tokens: 484 | latency: 2.85s
[14/30] done | tokens: 371 | latency: 2.34s
[15/30] done | tokens: 769 | latency: 5.54s
[16/30] done | tokens: 467 | latency: 3.10s
[17/30] done | tokens: 406 | latency: 2.69s
[18/30] done | tokens: 501 | latency: 3.67s
[19/30] done | tokens: 308 | latency: 2.16s
[20/30] done | tokens: 715 | latency: 4.56s
[21/30] done | tokens: 121 | latency: 0.80